# Final Project — Orchestrating a Penetration Test of Metasploitable 2
### Applied AI for Cybersecurity · comprehensive course project (2 weeks)

You will run a **real** (authorized, lab-only) penetration test of a **Metasploitable 2** VM and automate the analysis with the three approaches from this course — **Chains → RAG → Agents + MCP** — exactly the arc of `cybersecurity_automation.ipynb`, but on real scan data instead of simulated hosts.

**This notebook is your starting scaffold.** Develop on top of it: replace the simulated target with your VM's IP, run a real `nmap`, then **expand the RAG knowledge base(s)** so the assistant can ground the vulnerabilities you actually find, add **tools beyond nmap**, and write a **comparative report**.

> 📋 Follow **`FINAL_PROJECT_REQUIREMENTS.md`** for the full task list, deliverables, and rubric. The ✍️ **TODO** cells below mark what you must build.

> 🤝 **Two weeks.** Individual, or groups of up to 3 (scope scales with group size — see the requirements).

## ⚠️ Rules of Engagement (read first)

- Run scans/exploits **only** against your own Metasploitable 2 VM in an **isolated lab**.
- Configure the VM network as **Host-only** (or Internal) in VirtualBox — **never Bridged/NAT to the internet**. Metasploitable 2 is deliberately, severely vulnerable.
- Do not attack any other host. The CFAA and your university policy require staying in scope.
- Keep API keys in `.env` / Colab Secrets — never in code or in your report.

## 0 · Lab setup

1. Download **Metasploitable 2** and import it into **VirtualBox**.
2. Set its network adapter to **Host-only** (e.g., `vboxnet0`). Start the VM.
3. Find its IP: log in (`msfadmin`/`msfadmin`) and run `ip addr` (or `ifconfig`) — it's usually `192.168.56.x`. You can also discover it from your host with `nmap -sn 192.168.56.0/24`.
4. Install **nmap** on your host (and the other tools you choose in Part 3).

## 🛠️ Setup (imports)

In [ ]:
# pip install langchain-core sentence-transformers
# Optional (Stage 3 live agent): pip install mcp langchain-mcp-adapters langgraph langchain-openai
import re, subprocess
from typing import List
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage
print("Imports ready.")

## ✍️ TODO — 1 · Set your target IP

- Replace `REPLACE_ME` with your Metasploitable 2 VM's host-only IP.
- This is the step that turns the simulated lab into a real one.

In [ ]:
# ✍️ TODO: put your Metasploitable 2 VM's IP here (host-only network).
TARGET_IP = "REPLACE_ME"     # e.g. "192.168.56.101"
assert TARGET_IP != "REPLACE_ME", "Set TARGET_IP to your Metasploitable 2 VM's IP first."
print("Target:", TARGET_IP)

---
## 🔎 Stage 0 — Real recon with nmap

Run a service/version scan of your VM. `nmap -sV` gives the `port/service version` lines the rest of the pipeline parses — the **real** equivalent of the simulated scan in `cybersecurity_automation.ipynb`.

In [ ]:
def run_nmap(target: str) -> str:
    """Run `nmap -sV` against an authorized lab target and return its output.
    Requires nmap installed on your machine. If it is not, use the paste-fallback below."""
    try:
        out = subprocess.run(["nmap", "-sV", "-Pn", target],
                             capture_output=True, text=True, timeout=600)
        return out.stdout or out.stderr
    except FileNotFoundError:
        return "(nmap not found — install it, or paste your scan into NMAP_OUTPUT below)"

NMAP_OUTPUT = run_nmap(TARGET_IP)

# Fallback: if you ran nmap in a terminal instead, paste its output here (and comment the line above):
# NMAP_OUTPUT = """PORT   STATE SERVICE VERSION
# 21/tcp open ftp vsftpd 2.3.4
# ... (paste the rest) ..."""

print(NMAP_OUTPUT)

Parse the scan into a list of `service version` strings (the findings the assistant will reason about).

In [ ]:
def parse_services(scan_output: str) -> List[str]:
    """Pull 'service version' strings out of nmap-style lines like
    '21/tcp open ftp vsftpd 2.3.4'."""
    services = []
    for line in scan_output.splitlines():
        toks = line.split()
        if len(toks) >= 4 and "/" in toks[0] and toks[1] == "open":
            services.append(" ".join(toks[3:]))      # everything after the service name = product + version
    return services

services = parse_services(NMAP_OUTPUT)
print(f"Parsed {len(services)} services:")
for s in services:
    print("  -", s)

---
## 🔗 Stage 1 — Chains (LLM report from the scan)

The same chain as the worked example: parse → analyze → report. The stand-in model lets this run offline; swap in a real model later if you like. **No RAG yet — the model guesses.**

In [ ]:
def _analyst_from_memory(prompt_value):
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    items = [l.strip("- ").strip() for l in text.splitlines() if l.strip().startswith("-")]
    body = "From general knowledge, these services *may* have issues (UNVERIFIED guess, no CVE IDs):\n"
    for s in items:
        body += f"  - {s}: possibly outdated; could have public exploits.\n"
    return AIMessage(content=body + "Recommend manual verification.")

stage1_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a penetration-testing assistant. Analyze the services and write a short report."),
        ("user", "Services found:\n{services}")])
    | RunnableLambda(_analyst_from_memory) | StrOutputParser()
)

print("=== STAGE 1 REPORT (chain, LLM from memory) ===\n")
print(stage1_chain.invoke({"services": "\n".join(f"- {s}" for s in services)}))

---
## 🔍 Stage 2 — RAG (ground the analysis in real CVEs)

Index a CVE knowledge base and retrieve real entries per service. We start from a **copy** of the course KB so you can extend it freely.

In [ ]:
import shutil, os
# Work on YOUR OWN copy so you can expand it without touching the course file.
if not os.path.exists("project_cve_kb.md"):
    shutil.copy("cve_knowledge_base.md", "project_cve_kb.md")
    print("Created project_cve_kb.md (your editable copy). Expand THIS file.")
else:
    print("Using your project_cve_kb.md")

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

class STEmbeddings(Embeddings):
    def __init__(self, name="all-MiniLM-L6-v2"): self.m = SentenceTransformer(name)
    def embed_documents(self, t): return self.m.encode(t, normalize_embeddings=True).tolist()
    def embed_query(self, t):     return self.m.encode(t, normalize_embeddings=True).tolist()

def build_cve_store(path="project_cve_kb.md"):
    text = open(path, encoding="utf8").read()
    sections = [s.strip() for s in text.split("#") if s.strip()]
    docs = [Document(page_content=s, metadata={"title": s.splitlines()[0]}) for s in sections]
    store = InMemoryVectorStore(STEmbeddings()); store.add_documents(docs)
    return store, len(docs)

cve_store, n = build_cve_store()
print(f"Indexed {n} CVE entries from project_cve_kb.md\n")

def retrieve_cves(service, k=1, threshold=0.30):
    hits = cve_store.similarity_search_with_score(service, k=k)
    kept = [d.page_content for d, score in hits if score >= threshold]
    return kept[0] if kept else f"No known CVEs found for: {service}"

print("Coverage check — which services GROUND vs. hit a GAP:")
for s in services:
    line = retrieve_cves(s).splitlines()[0]
    flag = "GAP " if line.startswith("No known CVEs") else "ok  "
    print(f"  [{flag}] {s[:34]:36s} -> {line}")

## ✍️ TODO — 2 · Expand the CVE RAG (graded)

- Every service marked **GAP** above has no grounding. Research it and ADD an entry to `project_cve_kb.md` (one `# Title` section: what it is, the CVE/CWE, severity, remediation).
- Metasploitable 2 examples you will likely need: UnrealIRCd backdoor (CVE-2010-2075), distccd (CVE-2004-2687), Samba 3.0.20 usermap (CVE-2007-2447), PostgreSQL 8.3, Tomcat default creds, NFS no_root_squash, r-services (rexec/rlogin/rsh), ingreslock backdoor, Java RMI.
- Re-run the two cells above and show the **before/after**: GAPs that now ground.
- Then run the grounded report cell below.

In [ ]:
def _analyst_grounded(prompt_value):
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    context = text.split("# Retrieved CVEs", 1)[-1].split("Human:", 1)[0].strip()
    ids = sorted(set(re.findall(r"CVE-\d{4}-\d+", context)))
    head = f"GROUNDED REPORT — cites {len(ids)} CVE(s): {', '.join(ids)}\n\n"
    return AIMessage(content=head + context[:900] + (" ..." if len(context) > 900 else ""))

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a pentest assistant. Use ONLY the retrieved CVEs.\n# Retrieved CVEs\n{context}"),
    ("user", "Assess {target} given services:\n{services}")])

stage2_chain = (
    {"context": RunnableLambda(lambda t: "\n\n".join(retrieve_cves(s) for s in services)),
     "services": lambda t: "\n".join(f"- {s}" for s in services),
     "target": RunnableLambda(lambda t: t)}
    | rag_prompt | RunnableLambda(_analyst_grounded) | StrOutputParser()
)
print("=== STAGE 2 REPORT (RAG, grounded) ===\n")
print(stage2_chain.invoke(TARGET_IP))

---
## 🤖 Stage 3 — Agents + MCP (let the system decide)

An agent calls tools instead of following a fixed script. Start from the course `mcp_server.py` (it exposes `port_scan`, `lookup_cve`). The offline loop below shows the idea; the live agent (needs a key + the `mcp` SDK) is in `cybersecurity_automation.ipynb` / `web_pentest_automation.ipynb` — reuse that pattern, pointed at your expanded server.

In [ ]:
# Offline agent loop (deterministic) using your real scan + expanded RAG:
def tool_lookup_cve(service): return retrieve_cves(service).splitlines()[0]

def offline_agent(target, max_steps=30):
    print(f"GOAL: assess {target}\n")
    todo_services = list(services); report = []
    for step in range(1, max_steps + 1):
        if todo_services:
            s = todo_services.pop(0)
            print(f"  step {step}: REASON -> lookup_cve({s!r})")
            r = tool_lookup_cve(s); print(f"           OBSERVE -> {r[:70]}")
            report.append(r)
        else:
            print(f"  step {step}: REASON -> done"); break
    print("\n=== STAGE 3 REPORT (agent-orchestrated) ===")
    for r in report: print("  -", r)

offline_agent(TARGET_IP)

## ✍️ TODO — 3 · Add tools beyond nmap (graded)

- Pick at least TWO tools appropriate to Metasploitable 2, e.g. `nikto` (web), `enum4linux` / `smbclient` (SMB), `gobuster` (dirs), `searchsploit` (exploit-DB), `whatweb` (web fingerprint).
- Wrap each as a Python function (and ideally an MCP tool in your own copy of `mcp_server.py`) so the agent can call it — model it on the course server's `@mcp.tool()` functions.
- Run them against your VM and feed their findings into the pipeline.

## ✍️ TODO — 4 · Expand the RAG further / add a SECOND RAG file (graded)

- Your new tools produce new finding types (web paths, SMB shares, web-app CVEs). Add entries for them — either to `project_cve_kb.md` or, better, a SECOND file (e.g. `project_web_kb.md`, like the course `web_security_kb.md`).
- Build a second vector store for it (reuse `build_cve_store` with the new path) and retrieve from the right KB per finding type.

## ✍️ TODO — 5 · Maximize vulnerability coverage

- Iterate Stages 0–3 until your report covers as many of the VM's vulnerabilities as you can — map each to a CVE/CWE (and OWASP category for web findings).
- Aim for breadth AND correctness; note anything you could not ground and why.

---
## 📊 Compare the approaches (use this in your report)

In [ ]:
rows = [
    ("Orchestration",   "Chains",     "Chains",          "Agents"),
    ("Data source",     "LLM memory", "RAG (your KB)",   "RAG + MCP tools"),
    ("Accuracy",        "?",          "?",               "?"),     # ✍️ fill from YOUR results
    ("Effort to build", "?",          "?",               "?"),
    ("Scales to new tools", "?",      "?",               "?"),
    ("Autonomy",        "low",        "low",             "high"),
]
print(f"  {'CAPABILITY':20s}{'CHAINS':14s}{'RAG':18s}{'AGENT+MCP'}")
print("  " + "-" * 70)
for r in rows: print(f"  {r[0]:20s}{r[1]:14s}{r[2]:18s}{r[3]}")

## 📝 Deliverables (see FINAL_PROJECT_REQUIREMENTS.md for details & rubric)

1. **This notebook**, completed and runnable, with your real scan output and all TODOs done.
2. Your **expanded RAG file(s)**: `project_cve_kb.md` (+ a second KB if you added one).
3. Your **tool integrations** (and your edited `mcp_server.py` if you extended it).
4. A **comprehensive report** (PDF/MD): methodology, the vulnerabilities you found and grounded, your RAG-expansion process (before/after), and a **comparative analysis** of the three approaches to automating/orchestrating the pen-test.

> ⚠️ Keep everything in the isolated lab. Report only on your authorized VM.